### imports and pandas settings

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

In [2]:
pd.set_option("display.max_rows", None)

### set the path to data file

In [3]:
load_dotenv()

CF_OUTPUTS = Path(os.getenv("CF_OUTPUTS"))
print(CF_OUTPUTS)
print(CF_OUTPUTS.is_dir())

/home/dyretna/Dokument/Code/GitHub/nightingale_projects/cf_bench/cf_outputs
True


In [4]:
batch_data = CF_OUTPUTS / "xgb_optimized" / "XGBoost_thres0.1_2026-05-07" / "annotated.csv"

### load data 
convert to string to fillna with whitespace
we drop:  
- target_risk (half of "risk_before") , and 
- "hltprhc"

we rename "original" in df_id to "xin" (HL standards)

In [5]:
# Load CSV
batch_df = pd.read_csv(batch_data)

In [6]:
# Feature columns
feature_cols = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "bmi", "dosprt"]

# dtypes
int_columns = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "dosprt", "hltprhc", "Nchanged"]
float_columns = ["bmi", "gower_distance", "risk_before", "predicted_risk_after", "cf_gen_time_sec"]


# Convert and fill NaN with empty strings
batch_df[int_columns] = batch_df[int_columns].astype("Int64").astype("string").fillna("")
batch_df[float_columns] = batch_df[float_columns].round(4).astype("string").fillna("")
batch_df["valid"] = batch_df["valid"].astype("string").fillna("")

In [7]:
batch_df = batch_df.drop(columns=["hltprhc", "target_risk"])
batch_df["cf_id"] = batch_df["cf_id"].replace({"original": "xin"})

### All CFs

In [8]:
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,1.16,,,,,
1,0,cf_1,,,,7,,,18.9745,,,0.876,2,False,0.0743,0.0972
2,0,cf_2,,,6,,,,17.4904,,,1.0,2,False,0.0743,0.044
3,0,cf_3,,,,7,,,18.9745,7,,0.876,3,True,0.0743,0.0051
4,0,cf_4,,,,,1,,18.9619,5,,0.8771,3,True,0.0743,0.0022
5,0,cf_5,,,5,,,,18.9619,2,,0.8771,3,True,0.0743,0.0145
6,1,xin,3,4,1,2,3,0,22.3757,0,1.17,,,,,
7,1,cf_1,,,,,,,22.3757,,,0.8796,1,False,0.0484,0.0484
8,1,cf_2,,,2,,1,,22.3757,,,0.8796,3,False,0.0484,0.0534
9,1,cf_3,,,,3,1,,22.3757,,,0.8796,3,False,0.0484,0.0256


### Filtering Valid CFs

In [9]:
from cf_bench.utils import filter_valid_cfs

batch_df = filter_valid_cfs(batch_df)
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,1.16,,,,,
9,0,cf_3,,,,7,,,18.9745,7,,0.876,3,True,0.0743,0.0051
10,0,cf_4,,,,,1,,18.9619,5,,0.8771,3,True,0.0743,0.0022
11,0,cf_5,,,5,,,,18.9619,2,,0.8771,3,True,0.0743,0.0145
1,1,xin,3,4,1,2,3,0,22.3757,0,1.17,,,,,
12,1,cf_6,2,,,,1,,22.3757,,,0.8796,3,True,0.0484,0.0105
2,2,xin,5,3,1,1,4,0,22.694,7,1.25,,,,,
13,2,cf_3,,,,3,2,,22.6757,,,0.8755,3,True,0.0647,0.0185
14,2,cf_5,1,1,,,2,,22.68,,,0.8754,4,True,0.0647,0.0047
15,2,cf_6,4,,,4,2,,22.68,,,0.8754,4,True,0.0647,0.0063


### select one CF

In [10]:
from cf_bench.utils import select_one_cf_per_query

# Select one CF per observation (prefers valid CFs without violations)
single_cf_df = select_one_cf_per_query(batch_df, prefer_no_violations=True)
single_cf_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,1.16,,,,,
9,0,cf_3,,,,7,,,18.9745,7,,0.876,3,True,0.0743,0.0051
1,1,xin,3,4,1,2,3,0,22.3757,0,1.17,,,,,
10,1,cf_6,2,,,,1,,22.3757,,,0.8796,3,True,0.0484,0.0105
2,2,xin,5,3,1,1,4,0,22.694,7,1.25,,,,,
11,2,cf_5,1,1,,,2,,22.68,,,0.8754,4,True,0.0647,0.0047
3,3,xin,3,3,6,6,2,0,24.3418,1,1.3,,,,,
12,3,cf_1,,,,,,,24.3375,,,0.8834,1,False,0.0048,0.0048
4,4,xin,5,4,2,7,1,0,21.2585,3,1.39,,,,,
13,4,cf_5,4,,,,,,21.2585,,,0.9868,2,True,0.0259,0.007


In [11]:
from cf_bench.dice_adapters import DiceRecommender

recommender = DiceRecommender()

# Format a single query
for idx in range(0, 10):
    print(recommender.format_query(batch_df, query_index=idx))


=== Query index '0' ===
Task / Target: hltprhc
Query instance index: 0

Original instance:
  etfruit: 4
  eatveg: 3
  cgtsmok: 3
  alcfreq: 4
  slprl: 2
  paccnois: 0
  bmi: 18.9866
  dosprt: 0


=== Counterfactuals ===

--- cf_3 ---
Predicted risk: 0.0051
Valid: True
Changes:
  alcfreq: 4 → 7
  bmi: 18.9866 → 18.9745
  dosprt: 0 → 7

--- cf_4 ---
Predicted risk: 0.0022
Valid: True
Changes:
  slprl: 2 → 1
  bmi: 18.9866 → 18.9619
  dosprt: 0 → 5

--- cf_5 ---
Predicted risk: 0.0145
Valid: True
Changes:
  cgtsmok: 3 → 5
  bmi: 18.9866 → 18.9619
  dosprt: 0 → 2


=== Query index '1' ===
Task / Target: hltprhc
Query instance index: 1

Original instance:
  etfruit: 3
  eatveg: 4
  cgtsmok: 1
  alcfreq: 2
  slprl: 3
  paccnois: 0
  bmi: 22.3757
  dosprt: 0


=== Counterfactuals ===

--- cf_6 ---
Predicted risk: 0.0105
Valid: True
Changes:
  etfruit: 3 → 2
  slprl: 3 → 1


=== Query index '2' ===
Task / Target: hltprhc
Query instance index: 2

Original instance:
  etfruit: 5
  eatveg: 3
  c